In [ ]:
# conda activate genomic_tools

import os
import requests
import numpy as np
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed

In [ ]:
os.chdir("/mnt/lareaulab/reliscu/projects/NSF_GRFP/analyses/bulk/GTEx/cortex")

In [7]:
pd.set_option('display.max_columns', None)

In [ ]:
# Compile list of all genes from cell type splicing analysis
gene_list = []
for file in os.listdir("data/ctype_exons"):
    if file.endswith("exons.csv"):
        signif_exons_df = pd.read_csv(f"data/ctype_exons/{file}", index_col=0)
        signif_exons_df = signif_exons_df[~np.isnan(signif_exons_df['Oligo'])]  # keep as DataFrame, not just index
        gene_list.extend(signif_exons_df['Gene'].tolist())

genes = list(set(gene_list))  # unique genes 

In [ ]:
def chunks(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i:i+n]
        
results = {}
for chunk in chunks(genes, 1000):
    r = requests.post(
        "https://rest.ensembl.org/lookup/symbol/homo_sapiens",
        headers={"Content-Type": "application/json", "Accept": "application/json"},
        json={"symbols": chunk}
    )
    results.update(r.json())

df_gene_info = pd.DataFrame([
    {'gene': gene, 'description': info.get('description', 'N/A')}
    for gene, info in results.items()
    if isinstance(info, dict)
])

df_gene_info.to_csv("data/gene_descriptions.csv", index=False)

In [23]:
df_gene_info.head()

,gene,description
0,TM7SF2,transmembrane 7 superfamily member 2 [Source:H...
1,WDHD1,WD repeat and HMG-box DNA binding protein 1 [S...
2,MYLK,myosin light chain kinase [Source:HGNC Symbol;...
3,ZFYVE21,zinc finger FYVE-type containing 21 [Source:HG...
4,CHGB,chromogranin B [Source:HGNC Symbol;Acc:HGNC:1930]


In [ ]:
def get_gene_info(gene):
    try:
        r = requests.get(
            "https://rest.uniprot.org/uniprotkb/search",
            params={
                "query": f"gene:{gene} AND organism_id:9606 AND reviewed:true",
                "fields": "gene_names,protein_name,cc_function,keyword",
                "format": "json",
            },
            timeout=10  # give up after 10 seconds
        )
        if r.status_code == 200:
            hits = r.json().get('results', [])
            if hits:
                entry = hits[0]
                function = next((c['texts'][0]['value'] for c in entry.get('comments', []) if c['commentType'] == 'FUNCTION'), 'N/A')
                keywords = ', '.join([k['name'] for k in entry.get('keywords', [])])
                protein_name = entry.get('proteinDescription', {}).get('recommendedName', {}).get('fullName', {}).get('value', 'N/A')
                return {'gene': gene, 'protein_name': protein_name, 'function': function, 'keywords': keywords}
    except requests.exceptions.Timeout:
        pass
    return {'gene': gene, 'protein_name': 'N/A', 'function': 'N/A', 'keywords': 'N/A'}

from tqdm import tqdm

results = []
with ThreadPoolExecutor(max_workers=10) as executor:
    futures = {executor.submit(get_gene_info, gene): gene for gene in genes}
    for future in tqdm(as_completed(futures), total=len(genes)):
        results.append(future.result())

In [ ]:
df_uniprot = pd.DataFrame(results)
df_uniprot.to_csv("data/gene_uniprot_info.csv", index=False)